In [1]:
%%time
import pandas as pd  # Import pandas library for data manipulation
import numpy as np  # Import numpy library for numerical operations

# Define a constant for the Haraka rate (number of bytes processed per block)
HARAKAS_RATE = 32

# Function to load Haraka-specific constants (round constants)
def load_haraka_constants():
    # Define round constants as a 2D array of 32-bit unsigned integers
    """
    rc = np.array([
        [0x0684704c, 0xe620c00a, 0xb2c5fef0, 0x75817b9d],
        # Add more constants as needed
    ], dtype=np.uint32)
    """
    np.random.seed(42)  # Set a seed for reproducibility
    rc = np.random.randint(0, 0xFFFFFFFF, (4, 4), dtype=np.uint32)

    return rc  # Return the constants array

# Function to compute XOR (exclusive OR) of two 128-bit arrays
def xor128(a, b):
    return np.bitwise_xor(np.array(a, dtype=np.uint8), np.array(b, dtype=np.uint8))  # Perform XOR operation

# Function to absorb input data into the state array during Haraka computation
def haraka_S_absorb(s, r, m, mlen, p):
    # Convert input message `m` to a numpy array of bytes
    m = np.frombuffer(m, dtype=np.uint8)
    # Process full-rate blocks from the input message
    while mlen >= r:
        s[:16] = xor128(s[:16], m[:16])  # XOR first 16 bytes
        s[16:32] = xor128(s[16:32], m[16:32])  # XOR next 16 bytes
        mlen -= r  # Reduce the remaining length by the rate
        m = m[r:]  # Move to the next block

    # Handle the final partial block and padding
    t = np.zeros(r, dtype=np.uint8)  # Temporary buffer for padding
    t[:mlen] = m[:mlen]  # Copy remaining message bytes
    t[mlen] = p  # Add padding byte
    t[r-1] |= 128  # Add final padding bit
    s[:16] = xor128(s[:16], t[:16])  # XOR first 16 bytes
    s[16:32] = xor128(s[16:32], t[16:32])  # XOR next 16 bytes

# Function to compute the Haraka-S hash of an input message
def haraka_S(outlen, in_data, inlen):
    s = np.zeros(64, dtype=np.uint8)  # Initialize state array (64 bytes, all zeros)
    haraka_S_absorb(s, HARAKAS_RATE, in_data, inlen, 0x1F)  # Absorb input data into the state
    return s[:outlen]  # Return the first `outlen` bytes of the state as the hash

# Function to hash a single row or column of a dataset
def hash_row_or_column(data):
    data_bytes = data.to_string(index=False).encode()  # Convert row/column to bytes
    inlen = len(data_bytes)  # Calculate the length of the data in bytes
    return haraka_S(32, data_bytes, inlen)  # `32` specifies the desired output length

# Function to hash the entire dataset row-wise or column-wise
def hash_dataset(dataset):
    dataset_hashes = []  # Initialize a list to store hashes
    for _, row in dataset.iterrows():  # Iterate over each row in the dataset
        row_hash = hash_row_or_column(row)  # Compute hash for the row
        dataset_hashes.append(row_hash)  # Append the row hash to the list
    return np.array(dataset_hashes)  # Convert the list of hashes to a numpy array

# Example usage:
# Load dataset
df2 = pd.read_csv("elliptic_txs_features.csv")  # Load a CSV file into a pandas DataFrame
df = df2.sample(frac=0.025, random_state=42)  # Take a random 20% sample of the dataset

# Hash the dataset (row-wise)
dataset_hashes = hash_dataset(df)  # Compute hashes for all rows in the dataset

# Print original and hashed datasets
print("Original DataFrame:\n", df.head())  # Display only the first few rows of the original DataFrame
print("Dataset Hashes:\n", dataset_hashes)  # Display the computed hashes

# Create a mapping from hashes to original data
hash_to_data_mapping = {row_hash.tobytes(): row.tolist() for row, row_hash in zip(df.values, dataset_hashes)}

# Retrieve all data using the hash mapping
retrieved_data_all = [
    hash_to_data_mapping[row_hash.tobytes()] for row_hash in dataset_hashes
]

# Convert retrieved data back into a pandas DataFrame
retrieved_df = pd.DataFrame(retrieved_data_all, columns=df.columns)

# Print the retrieved DataFrame
print("Retrieved DataFrame:\n", retrieved_df.head())  # Display the first few rows of retrieved data


Original DataFrame:
         230425980   1  -0.1714692896288031  -0.18466755143291433  \
107272  272481041  24            -0.154168             -0.107012   
103928  331556016  23            -0.063761             -0.158783   
150374   29913073  37            -0.030351              0.138895   
89566   209062970  20            -0.172982             -0.175507   
159898   97388854  40            -0.160374              0.072863   

        -1.2013688016765636  -0.12196959975910057  -0.04387454791734898  \
107272            -1.201369              -0.12197             -0.043875   
103928            -1.201369              -0.12197             -0.043875   
150374            -0.091383              -0.12197             -0.043875   
89566              1.018602              -0.12197             -0.063725   
159898             1.018602              -0.12197             -0.043875   

        -0.11300200928476244  -0.06158379407303222  -0.16209679981659642  ...  \
107272             -0.113002          